# SparseLoCo + LoRA vs Baseline — Fine-tuning no MedQA (USMLE)

Comparação entre duas abordagens de fine-tuning do **Llama-2-7B** no benchmark **MedQA** (USMLE):

| Abordagem | Parâmetros treináveis | Comunicação |
|---|---|---|
| **Baseline** — AdamW + LoRA centralizado | LoRA adapters | sem compressão |
| **SparseLoCo + LoRA** — distribuído simulado | LoRA adapters | Top-k comprimido |

**Pergunta de pesquisa:** O SparseLoCo+LoRA atinge accuracy comparável ao LoRA centralizado no MedQA? Se sim, o SparseLoCo viabiliza fine-tuning distribuído sem degradar a qualidade do modelo.

> Modelo principal: `meta-llama/Llama-2-7b-hf` (requer login HuggingFace — ver célula 0)

In [ ]:
!pip install -q torch transformers peft datasets matplotlib tqdm scikit-learn bitsandbytes>=0.46.1 accelerate

## 0. Autenticação HuggingFace

> **⚠️ Obrigatório:** `meta-llama/Llama-2-7b-hf` é um modelo **gated** (acesso restrito pela Meta). Sem autenticação o download falha com erro 401.

### Passo 1 — Aceitar a licença da Meta (feito uma única vez)
1. Acesse `huggingface.co/meta-llama/Llama-2-7b-hf`
2. Clique em **"Expand to review and access model files"**
3. Preencha o formulário e aceite os termos — aprovação é automática e instantânea

### Passo 2 — Gerar um token de acesso (feito uma única vez)
1. Acesse `huggingface.co/settings/tokens`
2. Clique em **"New token"**
3. Escolha tipo **"Read"** (permissão mínima necessária)
4. Dê um nome (ex: `colab-llama`) e clique em **"Generate a token"**
5. Copie o token gerado (começa com `hf_...`)

### Passo 3 — Preencher a célula abaixo
Cole o token copiado na variável `HF_TOKEN` e execute a célula.

In [ ]:
from huggingface_hub import login

# ─────────────────────────────────────────────────────────────────────────────
# PREENCHA AQUI com seu token HuggingFace (huggingface.co/settings/tokens)
# Tipo "Read" é suficiente. Veja as instruções na célula acima.
# ─────────────────────────────────────────────────────────────────────────────
HF_TOKEN = ""  # ex: "hf_xxxxxxxxxxxxxxxxxxxx"

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Login realizado com sucesso.")
else:
    raise ValueError(
        "HF_TOKEN não preenchido. Preencha a variável acima com seu token "
        "HuggingFace antes de continuar — veja as instruções na célula anterior."
    )

In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset, Subset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Hiperparâmetros

In [ ]:
# Modelo
MODEL_ID = "meta-llama/Llama-2-7b-hf"
# Alternativa sem login (menor capacidade):
# MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Distribuição
R        = 4      # workers simulados
H        = 50     # inner steps por worker por round
T        = 30     # outer rounds

# Modelo e dados
B        = 4      # batch size
MAX_LEN  = 512   # cobre questões longas do MedQA
N_LABELS = 5      # opções A, B, C, D, E

# LoRA
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# Otimizadores
LR_INNER      = 2e-4
LR_OUTER      = 1.0
BETA          = 0.95
TOPK_K        = 0.20
MAX_GRAD_NORM = 1.0
WARMUP_STEPS  = 50

# Checkpoints (persiste entre sessões via Google Drive)
from google.colab import drive; drive.mount('/content/drive')
CKPT_DIR = "/content/drive/MyDrive/checkpoints"
# CKPT_DIR = "/content/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

SEED = 42
torch.manual_seed(SEED)

## 2. Dataset — MedQA (Domínio Biomédico)

MedQA é um benchmark de questões do exame médico americano (USMLE). Formato instruction-following com pergunta e 5 opções — a resposta é a letra correta (A, B, C, D, E).

- Input: instruction + questão com opções
- Labels: A→0, B→1, C→2, D→3, E→4 | Chance = 20%

In [ ]:
from sklearn.model_selection import train_test_split

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# IDs dos tokens de resposta que o modelo deve prever após "Answer:"
ANSWER_IDS = torch.tensor([
    tok.encode(f" {c}", add_special_tokens=False)[-1]
    for c in ["A", "B", "C", "D", "E"]
])
print(f"Answer token IDs: { {c: ANSWER_IDS[i].item() for i, c in enumerate('ABCDE')} }")

LABEL_MAP = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

raw = load_dataset("medalpaca/medical_meadow_medqa", split="train")
print(f"Total de amostras: {len(raw)}")

all_samples = [d for d in raw if str(d["output"]).strip()[0].upper() in LABEL_MAP]
print(f"Amostras válidas: {len(all_samples)}")

train_val, test_data = train_test_split(all_samples, test_size=1000, random_state=SEED)
train_data, val_data = train_test_split(train_val,   test_size=1000, random_state=SEED)
print(f"Treino: {len(train_data)} | Validação: {len(val_data)} | Teste: {len(test_data)}")

class MedQADataset(Dataset):
    def __init__(self, data):
        self.samples = data

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        item = self.samples[i]
        # Formato CausalLM: o modelo prediz a letra após "Answer:"
        prompt = f"{item['instruction']} {item['input']}\nAnswer:"
        enc = tok(prompt, truncation=True, max_length=MAX_LEN,
                  padding="max_length", return_tensors="pt")
        label = LABEL_MAP[str(item["output"]).strip()[0].upper()]
        return enc["input_ids"].squeeze(), enc["attention_mask"].squeeze(), \
               torch.tensor(label)

train_ds = MedQADataset(train_data)
val_ds   = MedQADataset(val_data)
test_ds  = MedQADataset(test_data)

per = len(train_ds) // R
shards = [
    DataLoader(Subset(train_ds, range(r*per, (r+1)*per)),
               batch_size=B, shuffle=True, drop_last=True)
    for r in range(R)
]
central_loader = DataLoader(train_ds, batch_size=B, shuffle=True, drop_last=True)
val_loader     = DataLoader(val_ds,   batch_size=B, shuffle=False)
test_loader    = DataLoader(test_ds,  batch_size=B, shuffle=False)

print(f"Chance level: {1/N_LABELS:.0%}")

## 3. Modelo — Llama-2-7B + QLoRA (CausalLM)

Llama-2-7B carregado com **4-bit quantization NF4** (QLoRA) usando abordagem **CausalLM**:
o modelo prediz a letra de resposta como próximo token após `"Answer:"`, sem nenhuma cabeça de classificação aleatória.

- Pesos frozen em 4-bit (~3.5GB)
- LoRA rank=16 aplicado em atenção e MLP (`task_type=CAUSAL_LM`)
- Avaliação: argmax dos logits restritos aos tokens `" A"`, `" B"`, `" C"`, `" D"`, `" E"`

In [ ]:
def make_model():
    if DEVICE == "cuda":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        base = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.bfloat16,
        )
        base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    else:
        base = AutoModelForCausalLM.from_pretrained(MODEL_ID)
        base = base.to(DEVICE)

    base.config.pad_token_id = tok.pad_token_id

    lora = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        bias="none",
    )
    model = get_peft_model(base, lora)
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        print(f"VRAM usada: {torch.cuda.memory_allocated()/1e9:.1f}GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")
    return model

def lora_params(m):
    return [p for p in m.parameters() if p.requires_grad]

def to_vec(params):
    return torch.cat([p.detach().cpu().float().flatten() for p in params])

def load_vec(v, params):
    off = 0
    for p in params:
        n = p.numel()
        p.data.copy_(v[off:off+n].view(p.shape).to(p.device).to(p.dtype))
        off += n

m_test = make_model()
m_test.print_trainable_parameters()

## 4. Funções de Treino e Avaliação

In [ ]:
def _answer_logits(logits, mask):
    """Logits dos tokens A-E na última posição não-padding de cada amostra."""
    last_pos = mask.sum(dim=1) - 1                          # (B,)
    last = logits[torch.arange(logits.shape[0]), last_pos]  # (B, vocab)
    return last[:, ANSWER_IDS.to(logits.device)]            # (B, 5)

def step_loss(m, ids, mask, labels):
    out = m(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE))
    ans = _answer_logits(out.logits, mask.to(DEVICE))
    return F.cross_entropy(ans, labels.to(DEVICE))

@torch.no_grad()
def evaluate(m, loader):
    m.eval()
    preds, targets = [], []
    for ids, mask, labels in loader:
        out = m(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE))
        ans = _answer_logits(out.logits, mask.to(DEVICE))
        preds.extend(ans.argmax(-1).cpu().tolist())
        targets.extend(labels.tolist())
    m.train()
    return {
        "acc":         accuracy_score(targets, preds),
        "macro_f1":    f1_score(targets, preds, average="macro",    zero_division=0),
        "weighted_f1": f1_score(targets, preds, average="weighted", zero_division=0),
        "kappa":       cohen_kappa_score(targets, preds),
        "preds":       preds,
        "targets":     targets,
    }

def topk_sparsify(v, k):
    out = torch.zeros_like(v)
    idx = v.abs().topk(k).indices
    out[idx] = v[idx]
    return out

## 5. Treinamento — SparseLoCo + LoRA

Cada round:
1. R workers treinam H steps localmente a partir dos parâmetros globais
2. Pseudo-gradiente: Δᵣ = θ_global − θ_local
3. Error Feedback + Top-k compressão
4. Outer update: θ_global ← θ_global − α · média(Δ̂ᵣ)

In [ ]:
def run_sparseloco(m, shards, val_loader):
    ps = lora_params(m)
    d  = sum(p.numel() for p in ps)
    k  = max(1, int(d * TOPK_K))

    ckpt_path        = f"{CKPT_DIR}/sparseloco_ckpt.pt"
    total_steps      = T * H
    bytes_per_round  = R * k * 4  # float32 = 4 bytes por param

    if os.path.exists(ckpt_path):
        print(f"Retomando checkpoint: {ckpt_path}")
        ckpt         = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        start_t      = ckpt["round"] + 1
        load_vec(ckpt["lora_params"], ps)
        ef           = ckpt["ef"]
        opt_states   = ckpt["opt_states"]
        sched_states = ckpt.get("sched_states", [None] * R)
        history      = ckpt["history"]
        cum_bytes    = history[-1].get("bytes_tx", 0) if history else 0
        print(f"Continuando do round {start_t}/{T}")
    else:
        start_t      = 0
        ef           = [torch.zeros(d) for _ in range(R)]
        opt_states   = [None] * R
        sched_states = [None] * R
        history      = []
        cum_bytes    = 0

    opts   = [torch.optim.AdamW(ps, lr=LR_INNER) for _ in range(R)]
    scheds = [get_cosine_schedule_with_warmup(opts[r], WARMUP_STEPS, total_steps)
              for r in range(R)]
    print(f"LoRA params: {d:,} | Top-k: k={k} ({TOPK_K*100:.0f}%)")

    for t in tqdm(range(start_t, T), desc="SparseLoCo"):
        g_vec      = to_vec(ps)
        local_vecs = []
        round_loss = 0.0

        for r in range(R):
            load_vec(g_vec.clone(), ps)
            if opt_states[r]   is not None: opts[r].load_state_dict(opt_states[r])
            if sched_states[r] is not None: scheds[r].load_state_dict(sched_states[r])
            it = iter(shards[r])
            for _ in range(H):
                try:    ids, mask, labels = next(it)
                except: it = iter(shards[r]); ids, mask, labels = next(it)
                opts[r].zero_grad()
                loss = step_loss(m, ids, mask, labels)
                loss.backward()
                opts[r].step()
                scheds[r].step()
                round_loss += loss.item()
            opt_states[r]   = opts[r].state_dict()
            sched_states[r] = scheds[r].state_dict()
            local_vecs.append(to_vec(ps))

        agg = torch.zeros(d)
        for r, lv in enumerate(local_vecs):
            pg     = g_vec - lv
            ef[r]  = BETA * ef[r] + pg
            comp   = topk_sparsify(ef[r], k)
            ef[r] -= comp
            agg   += comp
        agg /= R

        load_vec(g_vec - LR_OUTER * agg, ps)

        cum_bytes += bytes_per_round
        metrics    = evaluate(m, val_loader)
        ef_norm    = ef[0].norm().item()
        history.append({
            "train_loss":  round_loss / (R * H),
            "val_acc":     metrics["acc"],
            "macro_f1":    metrics["macro_f1"],
            "weighted_f1": metrics["weighted_f1"],
            "kappa":       metrics["kappa"],
            "ef_norm":     ef_norm,
            "bytes_tx":    cum_bytes,
        })
        tqdm.write(
            f"  round={t:2d}  loss={history[-1]['train_loss']:.4f}"
            f"  acc={metrics['acc']:.4f}  f1={metrics['macro_f1']:.4f}"
            f"  κ={metrics['kappa']:.4f}  ef={ef_norm:.3f}"
        )

        torch.save({
            "round":        t,
            "lora_params":  to_vec(ps),
            "ef":           ef,
            "opt_states":   opt_states,
            "sched_states": sched_states,
            "history":      history,
        }, ckpt_path)

    return history

## 6. Treinamento — Baseline (AdamW + LoRA centralizado)

In [ ]:
def run_baseline(m, loader, val_loader):
    ps          = lora_params(m)
    d           = sum(p.numel() for p in ps)
    total_steps = T * R * H  # mesmo número de steps que SparseLoCo no total

    ckpt_path       = f"{CKPT_DIR}/baseline_ckpt.pt"
    bytes_per_round = R * d * 4  # baseline transmite todos os params

    if os.path.exists(ckpt_path):
        print(f"Retomando checkpoint: {ckpt_path}")
        ckpt    = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        start_t = ckpt["round"] + 1
        load_vec(ckpt["lora_params"], ps)
        opt   = torch.optim.AdamW(ps, lr=LR_INNER)
        sched = get_cosine_schedule_with_warmup(opt, WARMUP_STEPS, total_steps)
        opt.load_state_dict(ckpt["opt_state"])
        sched.load_state_dict(ckpt["sched_state"])
        history   = ckpt["history"]
        cum_bytes = history[-1].get("bytes_tx", 0) if history else 0
        print(f"Continuando do round {start_t}/{T}")
    else:
        start_t = 0
        opt     = torch.optim.AdamW(ps, lr=LR_INNER)
        sched   = get_cosine_schedule_with_warmup(opt, WARMUP_STEPS, total_steps)
        history   = []
        cum_bytes = 0

    it              = iter(loader)
    steps_per_round = R * H

    for t in tqdm(range(start_t, T), desc="Baseline"):
        acc_loss = 0.0
        for _ in range(steps_per_round):
            try:    ids, mask, labels = next(it)
            except: it = iter(loader); ids, mask, labels = next(it)
            opt.zero_grad()
            loss = step_loss(m, ids, mask, labels)
            loss.backward()
            opt.step()
            sched.step()
            acc_loss += loss.item()

        cum_bytes += bytes_per_round
        metrics = evaluate(m, val_loader)
        history.append({
            "train_loss":  acc_loss / steps_per_round,
            "val_acc":     metrics["acc"],
            "macro_f1":    metrics["macro_f1"],
            "weighted_f1": metrics["weighted_f1"],
            "kappa":       metrics["kappa"],
            "bytes_tx":    cum_bytes,
        })
        tqdm.write(
            f"  round={t:2d}  loss={history[-1]['train_loss']:.4f}"
            f"  acc={metrics['acc']:.4f}  f1={metrics['macro_f1']:.4f}"
            f"  κ={metrics['kappa']:.4f}  lr={sched.get_last_lr()[0]:.2e}"
        )

        torch.save({
            "round":       t,
            "lora_params": to_vec(ps),
            "opt_state":   opt.state_dict(),
            "sched_state": sched.state_dict(),
            "history":     history,
        }, ckpt_path)

    return history

## 7. Executar Experimentos

In [ ]:
print('\n=== SparseLoCo + LoRA ===')
m_sloco = make_model()
hist_sloco = run_sparseloco(m_sloco, shards, val_loader)
del m_sloco
if DEVICE == "cuda": torch.cuda.empty_cache()

print('\n=== Baseline: AdamW + LoRA ===')
m_base = make_model()
hist_base = run_baseline(m_base, central_loader, val_loader)

## 8. Resultados

In [ ]:
# ── Zero-shot: modelo base sem nenhum fine-tuning ────────────────────────────
print('=== Zero-shot (sem fine-tuning) ===')
m_zero = make_model()
zero_val_metrics  = evaluate(m_zero, val_loader)
zero_test_metrics = evaluate(m_zero, test_loader)
print(f"  val  | acc={zero_val_metrics['acc']:.4f}  macro_f1={zero_val_metrics['macro_f1']:.4f}  κ={zero_val_metrics['kappa']:.4f}")
print(f"  test | acc={zero_test_metrics['acc']:.4f}  macro_f1={zero_test_metrics['macro_f1']:.4f}  κ={zero_test_metrics['kappa']:.4f}")
del m_zero
if DEVICE == "cuda": torch.cuda.empty_cache()

In [ ]:
def _extract(hist, key):
    return [h[key] for h in hist]

sloco_loss    = _extract(hist_sloco, "train_loss")
sloco_acc     = _extract(hist_sloco, "val_acc")
sloco_f1      = _extract(hist_sloco, "macro_f1")
sloco_kappa   = _extract(hist_sloco, "kappa")
sloco_efnorm  = _extract(hist_sloco, "ef_norm")
sloco_bytes   = _extract(hist_sloco, "bytes_tx")

base_loss     = _extract(hist_base, "train_loss")
base_acc      = _extract(hist_base, "val_acc")
base_f1       = _extract(hist_base, "macro_f1")
base_kappa    = _extract(hist_base, "kappa")
base_bytes    = _extract(hist_base, "bytes_tx")

# ── Figura 1: curvas de treinamento ──────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

ax = axes[0, 0]
ax.plot(sloco_loss, "o-", ms=4, label="SparseLoCo+LoRA")
ax.plot(base_loss,  "s-", ms=4, label="AdamW+LoRA")
ax.set_xlabel("Outer round"); ax.set_ylabel("Training loss")
ax.set_title("Training Loss"); ax.legend()

ax = axes[0, 1]
ax.plot(sloco_acc, "o-", ms=4, label="SparseLoCo+LoRA")
ax.plot(base_acc,  "s-", ms=4, label="AdamW+LoRA")
ax.axhline(zero_val_metrics["acc"], color="gray",      ls=":", label=f"Zero-shot ({zero_val_metrics['acc']:.3f})")
ax.axhline(1/N_LABELS,              color="lightgray", ls="--", alpha=0.7, label=f"Chance ({1/N_LABELS:.0%})")
ax.set_xlabel("Outer round"); ax.set_ylabel("Accuracy")
ax.set_title("Validation Accuracy"); ax.legend(); ax.set_ylim(0.1, 1.0)

ax = axes[0, 2]
ax.plot(sloco_f1, "o-", ms=4, label="SparseLoCo+LoRA")
ax.plot(base_f1,  "s-", ms=4, label="AdamW+LoRA")
ax.axhline(zero_val_metrics["macro_f1"], color="gray", ls=":", label=f"Zero-shot ({zero_val_metrics['macro_f1']:.3f})")
ax.set_xlabel("Outer round"); ax.set_ylabel("Macro F1")
ax.set_title("Macro F1"); ax.legend()

ax = axes[1, 0]
ax.plot(sloco_bytes, sloco_acc, "o-", ms=4, label=f"SparseLoCo+LoRA (Top-{TOPK_K*100:.0f}%)")
ax.plot(base_bytes,  base_acc,  "s-", ms=4, label="AdamW+LoRA (baseline)")
ax.axhline(zero_val_metrics["acc"], color="gray", ls=":", label="Zero-shot")
ax.set_xlabel("Bytes transmitidos (acumulado)"); ax.set_ylabel("Val Accuracy")
ax.set_title("Eficiência de Comunicação"); ax.legend()

ax = axes[1, 1]
ax.plot(sloco_efnorm, "o-", ms=4, color="tab:orange")
ax.set_xlabel("Outer round"); ax.set_ylabel("‖ef‖")
ax.set_title("Error Feedback Norm (SparseLoCo)")

ax = axes[1, 2]
ax.plot(sloco_kappa, "o-", ms=4, label="SparseLoCo+LoRA")
ax.plot(base_kappa,  "s-", ms=4, label="AdamW+LoRA")
ax.axhline(zero_val_metrics["kappa"], color="gray", ls=":", label=f"Zero-shot ({zero_val_metrics['kappa']:.3f})")
ax.set_xlabel("Outer round"); ax.set_ylabel("Cohen's κ")
ax.set_title("Cohen's Kappa"); ax.legend()

plt.suptitle(f'SparseLoCo+LoRA vs AdamW+LoRA — {MODEL_ID.split("/")[-1]} | MedQA  (R={R}, H={H}, β={BETA})')
plt.tight_layout()
plt.savefig("resultados.png", dpi=150)
plt.show()

# ── Avaliação final no test set ──────────────────────────────────────────────
# m_sloco foi deletado para liberar VRAM — recarrega do checkpoint
print("Carregando SparseLoCo do checkpoint para avaliação no test set...")
m_sloco_eval = make_model()
ckpt_sloco   = torch.load(f"{CKPT_DIR}/sparseloco_ckpt.pt", map_location="cpu", weights_only=False)
load_vec(ckpt_sloco["lora_params"], lora_params(m_sloco_eval))
test_sloco = evaluate(m_sloco_eval, test_loader)
del m_sloco_eval
if DEVICE == "cuda": torch.cuda.empty_cache()

test_base = evaluate(m_base, test_loader)

# ── Figura 2: confusion matrices ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
labels_str = list("ABCDE")
for ax, metrics, title in zip(
    axes,
    [zero_test_metrics, test_sloco, test_base],
    ["Zero-shot", "SparseLoCo+LoRA", "AdamW+LoRA"],
):
    cm = confusion_matrix(metrics["targets"], metrics["preds"], labels=list(range(N_LABELS)))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(N_LABELS)); ax.set_xticklabels(labels_str)
    ax.set_yticks(range(N_LABELS)); ax.set_yticklabels(labels_str)
    ax.set_xlabel("Predito"); ax.set_ylabel("Real"); ax.set_title(title)
    for i in range(N_LABELS):
        for j in range(N_LABELS):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.suptitle("Confusion Matrix — Test Set")
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()

# ── Figura 3: per-class F1 ────────────────────────────────────────────────────
per_class = {
    name: f1_score(m["targets"], m["preds"], average=None, labels=list(range(N_LABELS)), zero_division=0)
    for name, m in [("Zero-shot", zero_test_metrics), ("SparseLoCo+LoRA", test_sloco), ("AdamW+LoRA", test_base)]
}
x = np.arange(N_LABELS); width = 0.25
fig, ax = plt.subplots(figsize=(9, 4))
for i, (name, vals) in enumerate(per_class.items()):
    ax.bar(x + i * width, vals, width, label=name)
ax.set_xticks(x + width); ax.set_xticklabels(labels_str)
ax.set_ylabel("F1"); ax.set_title("F1 por Classe — Test Set"); ax.legend()
plt.tight_layout()
plt.savefig("per_class_f1.png", dpi=150)
plt.show()

# ── Tabela de resultados finais ───────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"{'Método':<25} {'Acc':>6} {'MacroF1':>8} {'WtF1':>7} {'Kappa':>7} {'Bytes TX':>12}")
print(f"{'-'*70}")
rows = [
    ("Zero-shot",            zero_test_metrics, "0"),
    ("SparseLoCo+LoRA",      test_sloco,        f"{sloco_bytes[-1]/1e9:.1f}G"),
    ("AdamW+LoRA (baseline)",test_base,         f"{base_bytes[-1]/1e9:.1f}G"),
    ("Chance level",         {"acc":1/N_LABELS, "macro_f1":float("nan"), "weighted_f1":float("nan"), "kappa":0.0}, "—"),
]
for name, m, tx in rows:
    print(f"{name:<25} {m['acc']:>6.4f} {m['macro_f1']:>8.4f} {m['weighted_f1']:>7.4f} {m['kappa']:>7.4f} {tx:>12}")
print(f"{'='*70}")